# 03c — Cluster-Range Label Learning

**Purpose:** Generate `train_labels.csv` and `test_labels.csv` for downstream
classification notebooks (04 and 05).

**Method:** For each cluster and each biomarker, learn a 5th / 95th percentile
threshold from **train data only**, then apply those thresholds to both train and test.

**Run order:** 01_EDA → 02_Preprocessing → 03_Clustering → **03c** → 04 → 05

**Inputs:**
- `data/processed/train_wide_unscaled.csv`
- `data/processed/test_wide_unscaled.csv`
- `data/processed/train_clusters.csv`
- `data/processed/test_clusters.csv`

**Outputs:**
- `data/processed/train_labels.csv`
- `data/processed/test_labels.csv`
- `results/metrics/03c_cluster_thresholds.csv`

In [1]:
import os, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

PROCESSED = '../data/processed'
METRICS   = '../results/metrics'
os.makedirs(PROCESSED, exist_ok=True)
os.makedirs(METRICS,   exist_ok=True)

LABELS = ['Anaemia', 'Diabetes_Risk', 'Dyslipidemia',
          'Kidney_Risk', 'Liver_Stress', 'Thyroid_Abnormal']

MIN_CLUSTER_SAMPLES = 30   # fall back to global threshold below this
LOW_PCT  = 5               # percentile for low-abnormal thresholds
HIGH_PCT = 95              # percentile for high-abnormal thresholds

print('Config ready.')

Config ready.


## Biomarker Groups

Each group maps a label to the biomarker terms to search (case-insensitive substring).
- `high`: abnormal if value **>** high threshold
- `low`:  abnormal if value **<** low threshold
- `both`: abnormal if value < low threshold **or** > high threshold

In [2]:
# direction: 'high' | 'low' | 'both'
BIOMARKER_GROUPS = {
    'Anaemia': [
        {'terms': ['haemoglobin','hemoglobin','hgb'], 'direction': 'low'},
        {'terms': ['rbc'],                            'direction': 'low'},
        {'terms': ['mcv'],                            'direction': 'low'},
        {'terms': ['mch'],                            'direction': 'low'},
        {'terms': ['mchc'],                           'direction': 'low'},
    ],
    'Diabetes_Risk': [
        {'terms': ['hba1c','hemoglobin a1c','glycated'], 'direction': 'high'},
        {'terms': ['glucose','fasting glucose','blood sugar'], 'direction': 'high'},
    ],
    'Dyslipidemia': [
        {'terms': ['total cholesterol','cholesterol'], 'direction': 'high'},
        {'terms': ['ldl'],                             'direction': 'high'},
        {'terms': ['triglyceride','triglycerides'],    'direction': 'high'},
        {'terms': ['vldl'],                            'direction': 'high'},
        {'terms': ['hdl'],                             'direction': 'low'},
    ],
    'Kidney_Risk': [
        {'terms': ['creatinine'],                      'direction': 'high'},
        {'terms': ['bun','blood urea nitrogen'],        'direction': 'high'},
        {'terms': ['urea'],                             'direction': 'high'},
        {'terms': ['uric acid'],                        'direction': 'high'},
        {'terms': ['egfr','gfr'],                       'direction': 'low'},
    ],
    'Liver_Stress': [
        {'terms': ['alt','sgpt'],                       'direction': 'high'},
        {'terms': ['ast','sgot'],                       'direction': 'high'},
        {'terms': ['ggt'],                              'direction': 'high'},
        {'terms': ['bilirubin'],                        'direction': 'high'},
        {'terms': ['alp','alkaline phosphatase'],        'direction': 'high'},
    ],
    'Thyroid_Abnormal': [
        {'terms': ['tsh'],         'direction': 'both'},
        {'terms': ['t3','ft3'],    'direction': 'both'},
        {'terms': ['t4','ft4'],    'direction': 'both'},
        {'terms': ['thyroid'],     'direction': 'both'},
    ],
}
print('Biomarker groups defined.')

Biomarker groups defined.


## Helper Functions

In [3]:
def find_cols(df, terms):
    """
    Return all column names in df whose lowercase name contains
    any of the given terms (case-insensitive substring match).
    Returns [] if none found.
    """
    matched = []
    for c in df.columns:
        c_lo = c.lower()
        if any(t.lower() in c_lo for t in terms):
            matched.append(c)
    return matched


def learn_thresholds(train_df, cluster_col='cluster_id'):
    """
    For each cluster x biomarker-group, compute 5th / 95th percentile
    from non-null train values.  Falls back to global percentile when
    fewer than MIN_CLUSTER_SAMPLES non-null values exist in that cluster.

    Returns a nested dict:
      thresholds[label][col] = {'low': float|None, 'high': float|None,
                                'direction': str, 'cluster': int|'global'}
    And a list of threshold records for the CSV export.
    """
    thresholds  = {lbl: {} for lbl in LABELS}
    thr_records = []
    skipped     = []
    
    # IMPROVEMENT: Explicitly save global and cluster thresholds for novelty comparison
    cluster_thr_records = []
    global_thr_records = []

    clusters = sorted(train_df[cluster_col].dropna().unique())

    for label, bm_list in BIOMARKER_GROUPS.items():
        for bm in bm_list:
            terms     = bm['terms']
            direction = bm['direction']
            cols      = find_cols(train_df, terms)

            if not cols:
                skipped.append({'label': label, 'terms': str(terms)})
                continue

            # global fallback (computed once across all train rows)
            global_vals = train_df[cols].stack().dropna()
            if len(global_vals) == 0:
                skipped.append({'label': label, 'terms': str(terms)})
                continue
            g_low  = float(np.percentile(global_vals, LOW_PCT))
            g_high = float(np.percentile(global_vals, HIGH_PCT))

            for col in cols:
                # Add to global records (only once per test)
                if not any(r['test'] == col for r in global_thr_records):
                    global_thr_records.append({
                        'test': col,
                        'lower_5th': g_low,
                        'upper_95th': g_high
                    })
                
                for clust in clusters:
                    mask  = train_df[cluster_col] == clust
                    vals  = train_df.loc[mask, col].dropna()

                    if len(vals) >= MIN_CLUSTER_SAMPLES:
                        t_low  = float(np.percentile(vals, LOW_PCT))
                        t_high = float(np.percentile(vals, HIGH_PCT))
                        src    = 'cluster'
                    else:
                        t_low  = g_low
                        t_high = g_high
                        src    = 'global_fallback'

                    key = (label, col, clust)
                    if key not in thresholds[label]:
                        thresholds[label][key] = {
                            'low': t_low, 'high': t_high,
                            'direction': direction, 'source': src
                        }
                    thr_records.append({
                        'label':     label,
                        'column':    col,
                        'cluster':   int(clust),
                        'direction': direction,
                        'low_pct':   round(t_low,  4),
                        'high_pct':  round(t_high, 4),
                        'source':    src,
                    })
                    
                    cluster_thr_records.append({
                        'test': col,
                        'cluster': int(clust),
                        'lower_5th': t_low,
                        'upper_95th': t_high
                    })

    return thresholds, pd.DataFrame(thr_records), skipped, pd.DataFrame(cluster_thr_records), pd.DataFrame(global_thr_records)


def apply_thresholds(df, thresholds, cluster_col='cluster_id'):
    """
    Apply learned cluster-specific thresholds to a dataframe.
    Returns a label DataFrame with columns:
      document_id, Anaemia, Diabetes_Risk, ...
    """
    label_rows = []

    for _, row in df.iterrows():
        clust = row.get(cluster_col)
        record = {'document_id': row['document_id']}

        for label, thr_map in thresholds.items():
            abnormal = False
            for (lbl, col, c), thr in thr_map.items():
                if lbl != label or c != clust:
                    continue
                if col not in row.index or pd.isna(row[col]):
                    continue
                val = row[col]
                d   = thr['direction']
                if d == 'high'  and val > thr['high']:
                    abnormal = True; break
                if d == 'low'   and val < thr['low']:
                    abnormal = True; break
                if d == 'both'  and (val < thr['low'] or val > thr['high']):
                    abnormal = True; break
            record[label] = int(abnormal)

        label_rows.append(record)

    return pd.DataFrame(label_rows, columns=['document_id'] + LABELS)


print('Helpers defined.')


Helpers defined.


## Step 1 — Load Data

In [4]:
# Verify input files exist
required = [
    f'{PROCESSED}/train_wide_unscaled.csv',
    f'{PROCESSED}/test_wide_unscaled.csv',
    f'{PROCESSED}/train_clusters.csv',
    f'{PROCESSED}/test_clusters.csv',
]
for fpath in required:
    if not os.path.exists(fpath):
        raise FileNotFoundError(
            f'Required input not found: {fpath}\n'
            'Run 02_Preprocessing.ipynb and 03_Clustering.ipynb first.'
        )

train_unscaled = pd.read_csv(f'{PROCESSED}/train_wide_unscaled.csv')
test_unscaled  = pd.read_csv(f'{PROCESSED}/test_wide_unscaled.csv')
train_clusters = pd.read_csv(f'{PROCESSED}/train_clusters.csv')
test_clusters  = pd.read_csv(f'{PROCESSED}/test_clusters.csv')

print(f'Train (unscaled): {train_unscaled.shape}')
print(f'Test  (unscaled): {test_unscaled.shape}')
print(f'Train clusters:   {train_clusters.shape}')
print(f'Test  clusters:   {test_clusters.shape}')

Train (unscaled): (79993, 109)
Test  (unscaled): (19999, 109)
Train clusters:   (79993, 2)
Test  clusters:   (19999, 2)


## Step 2 — Merge Cluster IDs

In [5]:
# Merge cluster_id into unscaled frames using document_id
train_df = train_unscaled.merge(train_clusters, on='document_id', how='left')
test_df  = test_unscaled.merge(test_clusters,   on='document_id', how='left')

n_missing_train = train_df['cluster_id'].isna().sum()
n_missing_test  = test_df['cluster_id'].isna().sum()

print(f'Train after merge: {train_df.shape}  '
      f'(missing cluster_id: {n_missing_train})')
print(f'Test  after merge: {test_df.shape}  '
      f'(missing cluster_id: {n_missing_test})')
print(f'Unique clusters in train: '
      f'{sorted(train_df["cluster_id"].dropna().unique())}')

Train after merge: (79993, 110)  (missing cluster_id: 0)
Test  after merge: (19999, 110)  (missing cluster_id: 0)
Unique clusters in train: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


## Step 3 — Learn Cluster-Specific Thresholds (Train Only)

For each cluster × biomarker column:
- low threshold  = 5th percentile of non-null train values in that cluster
- high threshold = 95th percentile of non-null train values in that cluster

**Fallback:** if fewer than 30 non-null values exist in a cluster, the global
train percentile is used instead.

In [6]:
print('Learning cluster thresholds from TRAIN only...')
thresholds, thr_df, skipped, cluster_thr_df, global_thr_df = learn_thresholds(train_df)

print(f'\nThresholds learned: {len(thr_df):,} records')
print(f'Unique (label, column, cluster) combos: '
      f'{thr_df[["label","column","cluster"]].drop_duplicates().shape[0]}')

cluster_src = thr_df['source'].value_counts()
print(f'\nSource breakdown:')
for src, cnt in cluster_src.items():
    print(f'  {src}: {cnt}')

if skipped:
    print(f'\nSkipped/missing biomarker groups ({len(skipped)}):')
    for s in skipped:
        print(f'  label={s["label"]}  terms={s["terms"]}')
else:
    print('\nNo biomarker groups skipped.')


Learning cluster thresholds from TRAIN only...

Thresholds learned: 355 records
Unique (label, column, cluster) combos: 265

Source breakdown:
  cluster: 355

No biomarker groups skipped.


In [7]:
# Preview sample thresholds
print('Sample thresholds (first 20 rows):')
print(thr_df.head(20).to_string(index=False))

Sample thresholds (first 20 rows):
  label     column  cluster direction  low_pct  high_pct  source
Anaemia Hemoglobin        0       low   7.1000    14.400 cluster
Anaemia Hemoglobin        1       low  11.8000    16.700 cluster
Anaemia Hemoglobin        2       low   9.2000    15.100 cluster
Anaemia Hemoglobin        3       low  10.8000    16.300 cluster
Anaemia Hemoglobin        4       low  11.0900    15.700 cluster
Anaemia  RBC Count        0       low   2.4395     5.150 cluster
Anaemia  RBC Count        1       low   4.0800     5.860 cluster
Anaemia  RBC Count        2       low   3.6000     5.590 cluster
Anaemia  RBC Count        3       low   4.0000     5.760 cluster
Anaemia  RBC Count        4       low   3.6000     5.682 cluster
Anaemia        MCV        0       low  76.4950   102.205 cluster
Anaemia        MCV        1       low  78.6000   102.000 cluster
Anaemia        MCV        2       low  67.4095    99.300 cluster
Anaemia        MCV        3       low  73.0000    95.90

## Step 4 — Apply Thresholds to Train and Test

Row-wise labelling: for each patient, check each relevant biomarker column
against the **cluster-specific** threshold learned from train.

In [8]:
print('Applying thresholds to TRAIN set...')
train_labels = apply_thresholds(train_df, thresholds)
print(f'  train_labels shape: {train_labels.shape}')

print('Applying thresholds to TEST set...')
test_labels  = apply_thresholds(test_df, thresholds)
print(f'  test_labels shape:  {test_labels.shape}')

Applying thresholds to TRAIN set...
  train_labels shape: (79993, 7)
Applying thresholds to TEST set...
  test_labels shape:  (19999, 7)


## Step 5 — Label Statistics

In [9]:
print('=== Positive % per Label (Train) ===')
for lbl in LABELS:
    pos = train_labels[lbl].sum()
    pct = train_labels[lbl].mean() * 100
    print(f'  {lbl:<20} {pos:>6,} positive  ({pct:.1f}%)')

print(f'\nTotal train patients labelled: {len(train_labels):,}')
print(f'Total test  patients labelled: {len(test_labels):,}')

# Patients with at least one positive label
train_any = (train_labels[LABELS].sum(axis=1) > 0).mean() * 100
test_any  = (test_labels[LABELS].sum(axis=1) > 0).mean() * 100
print(f'\nPatients with >= 1 positive label: '
      f'train {train_any:.1f}%  |  test {test_any:.1f}%')

=== Positive % per Label (Train) ===
  Anaemia              12,355 positive  (15.4%)
  Diabetes_Risk         6,546 positive  (8.2%)
  Dyslipidemia         16,551 positive  (20.7%)
  Kidney_Risk          15,390 positive  (19.2%)
  Liver_Stress         21,419 positive  (26.8%)
  Thyroid_Abnormal     19,663 positive  (24.6%)

Total train patients labelled: 79,993
Total test  patients labelled: 19,999

Patients with >= 1 positive label: train 68.2%  |  test 68.2%


## Step 6 — Save Outputs

In [10]:
# Save label CSVs
train_labels.to_csv(f'{PROCESSED}/train_labels.csv', index=False)
test_labels.to_csv( f'{PROCESSED}/test_labels.csv',  index=False)
print(f'Saved: {PROCESSED}/train_labels.csv')
print(f'Saved: {PROCESSED}/test_labels.csv')

# Save threshold table
thr_df.to_csv(f'{METRICS}/03c_cluster_thresholds.csv', index=False)
print(f'Saved: {METRICS}/03c_cluster_thresholds.csv')

# IMPROVEMENT: Save cluster and global thresholds separately
cluster_thr_df.to_csv(f'{METRICS}/cluster_thresholds.csv', index=False)
global_thr_df.to_csv(f'{METRICS}/global_thresholds.csv', index=False)
print(f'Saved: {METRICS}/cluster_thresholds.csv')
print(f'Saved: {METRICS}/global_thresholds.csv')

print(f'\nSummary:')
print(f'  train_labels.csv  : {len(train_labels):,} rows x {len(LABELS)} labels')
print(f'  test_labels.csv   : {len(test_labels):,} rows x {len(LABELS)} labels')
print(f'  cluster_thresholds: {len(thr_df):,} threshold records')
print(f'  Skipped groups    : {len(skipped)}')
print('\n03c_ClusterRangeLearning complete. Proceed to 04_Classification.')


Saved: ../data/processed/train_labels.csv
Saved: ../data/processed/test_labels.csv
Saved: ../results/metrics/03c_cluster_thresholds.csv
Saved: ../results/metrics/cluster_thresholds.csv
Saved: ../results/metrics/global_thresholds.csv

Summary:
  train_labels.csv  : 79,993 rows x 6 labels
  test_labels.csv   : 19,999 rows x 6 labels
  cluster_thresholds: 355 threshold records
  Skipped groups    : 0

03c_ClusterRangeLearning complete. Proceed to 04_Classification.
